# AIC System — Notebook 01: Build FAISS Index

**Purpose:** Load pre-extracted CLIP-32 `.npy` features from the AIC Kaggle dataset,
build a FAISS HNSW index for fast visual retrieval, and generate the
`keyframe_master.parquet` metadata table.

**Output files** (saved to `/kaggle/working/indexes/`):
- `faiss_visual.index` — FAISS HNSW index (512-dim CLIP-32 vectors)
- `keyframe_master.parquet` — Master metadata table (n, frame_idx, pts_time, ...)
- `faiss_ids_map.json` — int → keyframe_id mapping

> Upload `/kaggle/working/indexes/` as a new Kaggle Dataset after this runs.

In [ ]:
# ============================================================
# CELL 1: Setup — Clone GitHub repo & install dependencies
# ============================================================
import subprocess, sys, os

GITHUB_REPO = "https://github.com/YOUR_USERNAME/AIC_System.git"  # ← change this
BRANCH = "main"
REPO_DIR = "/kaggle/working/AIC_System"

# Clone repo
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    GITHUB_REPO, REPO_DIR], check=True)
    print(f"Cloned to {REPO_DIR}")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
    print(f"Updated {REPO_DIR}")

# Add to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install dependencies (quiet)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", f"{REPO_DIR}/requirements.txt"], check=True)
print("Dependencies installed.")

In [ ]:
# ============================================================
# CELL 2: Configure Paths
# ============================================================
import os
from pathlib import Path

# ---- Kaggle Dataset Slug ----
# Go to your Kaggle dataset → Settings → slug is in the URL
DATASET_SLUG = "your-username/aic-hcmc-data"  # ← change this

# Auto-detect dataset input path
INPUT_ROOT = Path("/kaggle/input")
DATASET_NAME = DATASET_SLUG.split("/")[-1]   # e.g. "aic-hcmc-data"
DATASET_PATH = INPUT_ROOT / DATASET_NAME

# Verify dataset is mounted
if not DATASET_PATH.exists():
    print(f"ERROR: Dataset not found at {DATASET_PATH}")
    print("Available datasets:")
    for p in INPUT_ROOT.iterdir():
        print(f"  {p}")
    raise FileNotFoundError(f"Mount the '{DATASET_NAME}' dataset in Notebook settings.")

# ---- Derived Paths (matching Kaggle dataset structure) ----
NP_DIR          = DATASET_PATH / "clip-features-32-aic25-b" / "clip-features-32"
MAP_KF_DIR      = DATASET_PATH / "map-keyframes-aic25-b1" / "map-keyframes"
KF_IMG_ROOT     = DATASET_PATH / "keyframes" / "keyframes"

# ---- Output paths ----
OUTPUT_DIR = Path("/kaggle/working/indexes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset path:       {DATASET_PATH}")
print(f"CLIP .npy dir:      {NP_DIR}")
print(f"Map-keyframes dir:  {MAP_KF_DIR}")
print(f"Keyframe images:    {KF_IMG_ROOT}")
print(f"Output dir:         {OUTPUT_DIR}")
print()

# Sanity check: count files
npy_count = len(list(NP_DIR.glob("*.npy"))) if NP_DIR.exists() else 0
csv_count = len(list(MAP_KF_DIR.glob("*.csv"))) if MAP_KF_DIR.exists() else 0
print(f"Found {npy_count} .npy files | {csv_count} .csv files")

In [ ]:
# ============================================================
# CELL 3: Verify GPU & FAISS
# ============================================================
import torch
import faiss

print(f"PyTorch: {torch.__version__}")
print(f"FAISS:   {faiss.__version__}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GB")
else:
    print("No GPU — running on CPU (FAISS will still work)")

from src.common.types import KeyframeMeta, TextualKISQuery
from src.common.enums import QueryType
print("AIC System imports: OK")

In [ ]:
# ============================================================
# CELL 4: Build FAISS Index from .npy files
# ============================================================
import time
from src.database.faiss_db import FaissDB

CLIP_DIM = 512  # CLIP-32 feature dimension

t0 = time.time()
faiss_db = FaissDB(dim=CLIP_DIM)
faiss_db.build_from_npy_files(
    npy_dir=str(NP_DIR),
    id_offset=0,
    normalize=True,    # Cosine similarity via inner product
)

FAISS_INDEX_PATH = str(OUTPUT_DIR / "faiss_visual.index")
faiss_db.save(FAISS_INDEX_PATH)

print(f"\nFAISS index built in {time.time() - t0:.1f}s")
print(f"Total vectors: {faiss_db.total_vectors:,}")
print(f"Saved: {FAISS_INDEX_PATH}")

In [ ]:
# ============================================================
# CELL 5: Build keyframe_master.parquet from CSV files
# ============================================================
from src.storage.metadata_store import MetadataStore

t0 = time.time()
store = MetadataStore(
    map_keyframes_root=str(MAP_KF_DIR),
    keyframes_image_root=str(KF_IMG_ROOT),
)

PARQUET_PATH = str(OUTPUT_DIR / "keyframe_master.parquet")
df = store.build(save_path=PARQUET_PATH)

MAP_PATH = str(OUTPUT_DIR / "faiss_ids_map.json")
store.export_faiss_ids_map(MAP_PATH)

print(f"\nMetadata built in {time.time() - t0:.1f}s")
print(f"Total keyframes: {store.total_keyframes:,}")
print(f"Total videos:    {len(store.video_ids):,}")
print()
df.head(10)

In [ ]:
# ============================================================
# CELL 6: Sanity Check — Test search with a random vector
# ============================================================
import numpy as np

# Load index fresh from disk
test_db = FaissDB(dim=CLIP_DIM)
test_db.load(FAISS_INDEX_PATH)

# Random query vector
query_vec = np.random.randn(CLIP_DIM).astype(np.float32)
faiss_ids, scores = test_db.search(query_vec, top_k=5)

print("Top-5 search results (random query):")
for rank, (fid, score) in enumerate(zip(faiss_ids, scores), 1):
    meta = store.get_by_faiss_id(int(fid))
    if meta:
        print(f"  [{rank}] {meta.keyframe_id} | frame_idx={meta.frame_idx} "
              f"| pts_time={meta.pts_time:.2f}s | score={score:.4f}")

# Cross-check counts
assert test_db.total_vectors == store.total_keyframes, \
    f"MISMATCH: {test_db.total_vectors} vectors vs {store.total_keyframes} metadata rows!"
print(f"\nSanity check PASSED: {test_db.total_vectors:,} vectors == {store.total_keyframes:,} metadata rows")

In [ ]:
# ============================================================
# CELL 7: Show output files and sizes
# ============================================================
print("Output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:40s} {size_mb:.1f} MB")

print()
print("Next steps:")
print("  1. Go to Kaggle → Datasets → Create new dataset")
print("  2. Upload contents of /kaggle/working/indexes/ as 'aic-hcmc-indexes'")
print("  3. In future notebooks, mount this dataset as input")
print("  4. Run kaggle_02_extract_ocr.ipynb for OCR features")